# Figure 4A — VSMC PCA (VSL / modified-VSL screen)

Reproduces the manuscript panel: *"Unbiased UMAP analysis shows that
transcriptional profile of VSMCs treated with mVSL is the closest to that of
cells at day 0."*

**Input**: `Fig4A_VSMC_FPKM.csv` — Cuffdiff FPKM, 11 samples spanning day 0,
day-40 VSL, and five VSL modifications (GEO libraries S63-S66, S69-S75;
EXP1 batch). Two additional EXP1 libraries sequenced in the same batch (VSL
with serum depletion, S67/S68) are not part of the published panel and are
excluded here.

**Pipeline**: same workflow as the Figure 2D notebook -- log10(FPKM+1), top
most-variable genes, PCA centered but not scaled, followed by UMAP on the
PCA coordinates.


In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.decomposition import PCA
import umap

RNG = 5
OUT = Path("figures"); OUT.mkdir(exist_ok=True)
INPUT = Path("../Fig4A_VSMC_EXP1/processed/Fig4A_VSMC_FPKM.csv")

data = pd.read_csv(INPUT, index_col=0)
by_gene = data.set_index("gene_official")
by_gene = by_gene[~by_gene.index.duplicated(keep="first")]
sample_cols = [c for c in by_gene.columns if c != "gene_official"]
expr = by_gene[sample_cols]
print(expr.shape, "genes x", len(sample_cols), "samples")

groups = pd.Series([re.sub(r"_rep\d+$", "", s) for s in sample_cols],
                    index=sample_cols, name="group")
groups.value_counts()

In [ ]:
NTOP = 1000

log_fpkm = np.log10(expr + 1)
detected = (expr > 0).sum(axis=1) >= 3
log_fpkm = log_fpkm[detected]
print(f"{detected.sum()} / {len(detected)} genes detected in >= 3 samples")

top_var = log_fpkm.var(axis=1).sort_values(ascending=False).head(NTOP).index
top500 = log_fpkm.loc[top_var]
print(f"using top {top500.shape[0]} most-variable genes for PCA")

In [ ]:
pca = PCA(n_components=min(8, top500.shape[1] - 1))
pcs = pca.fit_transform(top500.T.values)
pca_df = pd.DataFrame(pcs[:, :2], index=sample_cols, columns=["PC1", "PC2"])
pca_df["group"] = groups
pca_df.to_csv(OUT / "Fig4A_pca_coordinates.csv")

plt.figure(figsize=(4, 3.2))
plt.bar(range(1, len(pca.explained_variance_ratio_) + 1),
        pca.explained_variance_ratio_ * 100)
plt.xlabel("PC"); plt.ylabel("% variance explained")
plt.tight_layout()
plt.savefig(OUT / "Fig4A_pca_variance_ratio.png", dpi=200)
plt.show()

In [ ]:
GROUP_ORDER = ["D0_Ctrl_VSMC", "D40_Ctrl_VSMC", "D40_VSL_VSMC",
               "D40_VSL_FGF_depleted_VSMC", "D40_VSL_8Br_cAMP_VSMC",
               "D40_VSL_PDGF_inhibitor_VSMC", "D40_mVSL_VSMC"]
present = [g for g in GROUP_ORDER if g in pca_df["group"].unique()]

# fixed colour assignment matching the published panel (matplotlib tab10)
colors = {"D0_Ctrl_VSMC": "#1f77b4", "D40_Ctrl_VSMC": "#ff7f0e", "D40_VSL_VSMC": "#2ca02c",
          "D40_VSL_FGF_depleted_VSMC": "#9467bd", "D40_VSL_8Br_cAMP_VSMC": "#bcbd22",
          "D40_VSL_PDGF_inhibitor_VSMC": "#8c564b", "D40_mVSL_VSMC": "#17becf"}

fig, ax = plt.subplots(figsize=(7, 5.5))
for g in present:
    sub = pca_df[pca_df["group"] == g]
    ax.scatter(sub.PC1, sub.PC2, s=160, color=colors[g], label=g, edgecolor="white", linewidth=0.6)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
ax.set_title("VSMC, day 0 vs day 40 VSL screen (EXP1) -- PCA")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
fig.tight_layout()
fig.savefig(OUT / "Fig4A_VSMC_PCA.pdf", bbox_inches="tight")
fig.savefig(OUT / "Fig4A_VSMC_PCA.png", dpi=300, bbox_inches="tight")
plt.show()
print("wrote", OUT / "Fig4A_VSMC_PCA.png")

In [ ]:
n_neighbors = 3
min_dist = 0.5
reducer = umap.UMAP(n_neighbors=n_neighbors, min_dist=min_dist, random_state=RNG, init="random")
emb = reducer.fit_transform(pcs)
umap_df = pd.DataFrame(emb, index=sample_cols, columns=["UMAP1", "UMAP2"])
umap_df["group"] = groups
umap_df.to_csv(OUT / "Fig4A_umap_coordinates.csv")

fig, ax = plt.subplots(figsize=(7, 5.5))
for g in present:
    sub = umap_df[umap_df["group"] == g]
    ax.scatter(sub.UMAP1, sub.UMAP2, s=160, color=colors[g], label=g, edgecolor="white", linewidth=0.6)
ax.set_xlabel("UMAP1"); ax.set_ylabel("UMAP2")
ax.set_title(f"VSMC -- UMAP on PCA coordinates (n_neighbors={n_neighbors}, random_state={RNG})")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
fig.tight_layout()
fig.savefig(OUT / "Fig4A_VSMC_UMAP.pdf", bbox_inches="tight")
fig.savefig(OUT / "Fig4A_VSMC_UMAP.png", dpi=300, bbox_inches="tight")
plt.show()
print("wrote", OUT / "Fig4A_VSMC_UMAP.png")